# Clean Facility Data

## About this Notebook

### Objective

Clean, standardize, and prepare EPA facility-level emissions data for integration with satellite observations and socioeconomic indicators.

This notebook produces a structured, modeling-ready facility dataset that will later be merged with Sentinel-2 imagery, Sentinel-5P NO₂ data, and ADI metrics.

---

### Inputs

- Raw EPA National Emissions Inventory (NEI) facility data
- Facility identifiers (e.g., facility ID, FIPS, location coordinates)
- Reported emissions values (e.g., NOx tons per year)
- Optional: compliance or violation indicators

Primary data bucket:
`gs://msads-mba-capstone-team-1/Data/`

---

### Outputs

- Cleaned and filtered facility-level dataset
- Standardized emissions variables
- Validated latitude/longitude fields
- Exported CSV saved to GCS for downstream notebooks

Example output path:
`gs://msads-mba-capstone-team-1/Data/TrainingData/clean_facility_data.csv`

This dataset will later be joined with satellite-derived features and ADI data.

---

### Workflow Overview

1. Load raw NEI facility data  
2. Select relevant pollutant(s) (e.g., NOx)  
3. Clean and standardize emissions fields  
4. Validate and clean geospatial coordinates  
5. Filter to relevant geography (e.g., national or Illinois subset)  
6. Export cleaned facility dataset to GCS  

---

### Key Notes

- Reported emissions serve as the self-reported benchmark in the verification analysis.
- Units (e.g., tons/year) must be consistent across all records.
- Missing or invalid coordinates must be handled before spatial joins.
- Facility IDs must remain stable for downstream merging.
- Any filtering decisions (e.g., sector restrictions) should be documented clearly.

---

### Pipeline Position

CleanFacilityData → ExportS2SensorData / ExportS5SensorData → FusionNO2Model

This notebook establishes the facility-level ground truth used for emissions verification.

## Config

In [1]:
# Input Path
nei_path = "gs://msads-mba-capstone-team-1/Data/TrainingData/2021_NEI_Facility_summary.csv"

# Output paths
main_path_an = "gs://msads-mba-capstone-team-1/Data/TrainingData/nei_2021_IL_nox_for_analysis.csv"
main_path_cl = "gs://msads-mba-capstone-team-1/Data/TrainingData/nei_2021_IL_nox_for_model.csv"

## Load Data

In [2]:
import pandas as pd
import numpy as np

# Load NEI 2021
nei = pd.read_csv(
    nei_path
)

print("Shape:", nei.shape)
print("\nColumns:\n", nei.columns)
print("\nMissing values:\n", nei.isnull().sum().sort_values(ascending=False).head(15))

/var/tmp/ipykernel_147555/322091878.py:5: DtypeWarning: Columns (2,7,8,13,18) have mixed types. Specify dtype option on import or set low_memory=False.
  nei = pd.read_csv(


Shape: (2005169, 33)

Columns:
 Index(['state', 'fips state code', 'tribal name', 'fips code', 'county',
       'eis facility id', 'program system code', 'agency facility id',
       'tri facility id', 'company name', 'site name', 'primary naics code',
       'primary naics description', 'facility source type', 'site latitude',
       'site longitude', 'address', 'city', 'zip code', 'postal abbreviation',
       'reporting period', 'emissions operating type', 'pollutant code',
       'pollutant desc', 'pollutant type(s)', 'hap type', 'total emissions',
       'emissions uom', 'data set', 'outlier minimum', 'outlier maximum',
       'outlier?', 'national maximum'],
      dtype='object')

Missing values:
 tribal name             1997064
outlier minimum         1974619
outlier maximum         1972742
tri facility id         1754092
facility source type    1070619
hap type                 962136
company name             624258
national maximum         411702
agency facility id       268414

In [3]:
nei_copy = nei.copy()
nei_copy.head()

,state,fips state code,tribal name,fips code,county,eis facility id,program system code,agency facility id,tri facility id,company name,...,pollutant desc,pollutant type(s),hap type,total emissions,emissions uom,data set,outlier minimum,outlier maximum,outlier?,national maximum
0,CA,6.0,NaN,6007,Butte,111,CARB,41604748.0,NaN,"JOSIASSEN FARMS, INC.",...,Manganese,HAP,HAP-Metal,12.723980,LB,2021NEI_FinalV2,NaN,NaN,No,10000.0
1,CA,6.0,NaN,6007,Butte,111,CARB,41604748.0,NaN,"JOSIASSEN FARMS, INC.",...,Arsenic,HAP,HAP-Metal,0.136049,LB,2021NEI_FinalV2,NaN,NaN,No,1000.0
2,CA,6.0,NaN,6007,Butte,111,CARB,41604748.0,NaN,"JOSIASSEN FARMS, INC.",...,Elemental Carbon portion of PM2.5-PRI,Other,NaN,0.001374,TON,2021NEI_FinalV2,NaN,NaN,No,NaN
3,CA,6.0,NaN,6007,Butte,111,CARB,41604748.0,NaN,"JOSIASSEN FARMS, INC.",...,Nitrate portion of PM2.5-PRI,Other,NaN,0.000000,TON,2021NEI_FinalV2,NaN,NaN,No,NaN
4,CA,6.0,NaN,6007,Butte,111,CARB,41604748.0,NaN,"JOSIASSEN FARMS, INC.",...,Nitrogen Oxides,CAP,NaN,0.070000,TON,2021NEI_FinalV2,NaN,NaN,No,1000.0


## Filter and Clean Data

In [5]:
# Filter to Illinois
nei = nei[nei["postal abbreviation"] == "IL"].copy()

print("After IL filter:", nei.shape)
nei.head()

After IL filter: (90725, 33)


,state,fips state code,tribal name,fips code,county,eis facility id,program system code,agency facility id,tri facility id,company name,...,pollutant desc,pollutant type(s),hap type,total emissions,emissions uom,data set,outlier minimum,outlier maximum,outlier?,national maximum
26742,IL,17.0,NaN,17019,Champaign,558811,ILEPA,019813AAA,NaN,NaN,...,Ethyl Benzene,HAP,HAP-VOC,0.000000,LB,2021NEI_FinalV2,NaN,NaN,No,5000.0
26743,IL,17.0,NaN,17019,Champaign,558811,ILEPA,019813AAA,NaN,NaN,...,Acrolein,HAP,HAP-VOC,0.000000,LB,2021NEI_FinalV2,NaN,NaN,No,10000.0
26744,IL,17.0,NaN,17019,Champaign,558811,ILEPA,019813AAA,NaN,NaN,...,Toluene,HAP,HAP-VOC,0.050544,LB,2021NEI_FinalV2,NaN,NaN,No,200000.0
26745,IL,17.0,NaN,17019,Champaign,558811,ILEPA,019813AAA,NaN,NaN,...,Phenol,HAP,HAP-VOC,0.000000,LB,2021NEI_FinalV2,NaN,NaN,No,10000.0
26746,IL,17.0,NaN,17019,Champaign,558811,ILEPA,019813AAA,NaN,NaN,...,Hexane,HAP,HAP-VOC,26.757820,LB,2021NEI_FinalV2,NaN,NaN,No,100000.0


In [6]:
cols = ["address", "city", "zip code"]

# Total NA count per column
na_counts = nei[cols].isna().sum()

print("NA counts per column:")
print(na_counts)

NA counts per column:
address     0
city        0
zip code    0
dtype: int64


In [9]:
nei = nei[nei["pollutant desc"] == "Nitrogen Oxides"].copy()

print("After NOx filter:", nei.shape)
print(nei["pollutant desc"].unique())
nei.head()

After NOx filter: (2433, 33)
['Nitrogen Oxides']


,state,fips state code,tribal name,fips code,county,eis facility id,program system code,agency facility id,tri facility id,company name,...,pollutant desc,pollutant type(s),hap type,total emissions,emissions uom,data set,outlier minimum,outlier maximum,outlier?,national maximum
26771,IL,17.0,NaN,17019,Champaign,558811,ILEPA,019813AAA,NaN,NaN,...,Nitrogen Oxides,CAP,NaN,78.318483,TON,2021NEI_FinalV2,43.0,684.0,No,NaN
26785,IL,17.0,NaN,17031,Cook,558911,ILEPA,031012ADF,NaN,Na,...,Nitrogen Oxides,CAP,NaN,1.006086,TON,2021NEI_FinalV2,NaN,NaN,No,1000.0
26857,IL,17.0,NaN,17031,Cook,559311,ILEPA,031006AAC,60501WNSCR5824S,Na,...,Nitrogen Oxides,CAP,NaN,27.900600,TON,2021NEI_FinalV2,16.0,63.0,No,NaN
26893,IL,17.0,NaN,17031,Cook,559511,ILEPA,031012AEA,NaN,Na,...,Nitrogen Oxides,CAP,NaN,6.143930,TON,2021NEI_FinalV2,NaN,NaN,No,1000.0
26908,IL,17.0,NaN,17011,Bureau,559711,ILEPA,011085ABC,61356SCHLG121WR,Na,...,Nitrogen Oxides,CAP,NaN,1.370200,TON,2021NEI_FinalV2,NaN,NaN,No,1000.0


In [10]:
# Clean Emissions
nei["total emissions"] = pd.to_numeric(nei["total emissions"], errors="coerce")

nei = nei[nei["total emissions"] > 0].copy()

print("After emissions cleaning:", nei.shape)

After emissions cleaning: (2319, 33)


In [11]:
# Remove duplicates
print("Unique facilities:", nei["eis facility id"].nunique())
print("Total rows:", len(nei))

Unique facilities: 2319
Total rows: 2319


## Geocode Data - Find Lat/Lon and Geoid

In [12]:
import pandas as pd

# Ensure numeric (sometimes blanks are strings)
nei["site latitude"] = pd.to_numeric(nei["site latitude"], errors="coerce")
nei["site longitude"] = pd.to_numeric(nei["site longitude"], errors="coerce")

# Filter rows with missing coordinates
missing_coords_df = nei[
    nei["site latitude"].isna() |
    nei["site longitude"].isna()
].copy()

print("Total rows with missing coordinates:", len(missing_coords_df))

# Count unique facilities
print("Unique facilities with missing coords:",
      missing_coords_df["eis facility id"].nunique())

Total rows with missing coordinates: 0
Unique facilities with missing coords: 0


### Geocode Lat and Lon
All rows have coordinates but if they didn't we can run it through this code to find coordinates from the address.

In [13]:
import requests

def census_geocode(address):
    if pd.isna(address):
        return None, None

    url = "https://geocoding.geo.census.gov/geocoder/locations/onelineaddress"
    params = {
        "address": address,
        "benchmark": "Public_AR_Current",
        "format": "json"
    }

    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        data = r.json()

        matches = data["result"]["addressMatches"]
        if len(matches) == 0:
            return None, None

        coords = matches[0]["coordinates"]
        return coords["y"], coords["x"]  # lat, lon

    except Exception:
        return None, None

In [14]:
import pandas as pd

# Ensure numeric
nei["site latitude"] = pd.to_numeric(nei["site latitude"], errors="coerce")
nei["site longitude"] = pd.to_numeric(nei["site longitude"], errors="coerce")

# Identify rows that actually need geocoding
mask_missing = (
    nei["site latitude"].isna() |
    nei["site longitude"].isna()
)

print("Rows needing geocode:", mask_missing.sum())

# Apply geocoding ONLY to those rows
nei.loc[mask_missing, ["site latitude", "site longitude"]] = (
    nei.loc[mask_missing]
       .apply(lambda row: census_geocode(row["address"]), axis=1)
       .apply(pd.Series)
)

Rows needing geocode: 0


In [15]:
nei = nei.dropna(subset=["site latitude", "site longitude"]).copy()
nei.shape

(2319, 33)

In [16]:
# Check Lat/Lon
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Basic summary
print("Latitude summary:")
print(nei["site latitude"].describe())
print("\nLongitude summary:")
print(nei["site longitude"].describe())

# Illinois bounding box check
print("\nIllinois expected bounds:")
print("Latitude ~ 36.9 to 42.5")
print("Longitude ~ -91.5 to -87.0")

print("\nActual bounds:")
print("Lat min/max:", nei["site latitude"].min(), nei["site latitude"].max())
print("Lon min/max:", nei["site longitude"].min(), nei["site longitude"].max())

Latitude summary:
count    2319.000000
mean       41.038124
std         1.298943
min        37.016051
25%        40.273493
50%        41.672193
75%        41.922724
max        42.493870
Name: site latitude, dtype: float64

Longitude summary:
count    2319.000000
mean      -88.596253
std         0.903442
min       -91.410788
25%       -89.164203
50%       -88.253361
75%       -87.854981
max       -87.527626
Name: site longitude, dtype: float64

Illinois expected bounds:
Latitude ~ 36.9 to 42.5
Longitude ~ -91.5 to -87.0

Actual bounds:
Lat min/max: 37.016051 42.49387
Lon min/max: -91.410788 -87.527626


In [17]:
print(nei["site latitude"].astype(str).str.split(".").str[1].str.len().describe())
print(nei["site longitude"].astype(str).str.split(".").str[1].str.len().describe())

count    2319.000000
mean        5.870203
std         0.408036
min         2.000000
25%         6.000000
50%         6.000000
75%         6.000000
max         6.000000
Name: site latitude, dtype: float64
count    2319.000000
mean        5.873652
std         0.402741
min         2.000000
25%         6.000000
50%         6.000000
75%         6.000000
max         6.000000
Name: site longitude, dtype: float64


In [18]:
print("Missing lat:", nei["site latitude"].isna().sum())
print("Missing lon:", nei["site longitude"].isna().sum())

Missing lat: 0
Missing lon: 0


### Geocode Geoid

In [19]:
import geopandas as gpd

url = "https://www2.census.gov/geo/tiger/TIGER2020/BG/tl_2020_17_bg.zip"
bg = gpd.read_file(url)

bg.head()

,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,GEOID,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,17,043,841316,1,170438413161,Block Group 1,G5030,S,3149456,85825,+41.9776901,-088.1968528,"POLYGON ((-88.2074 41.98733, -88.20728 41.9873..."
1,17,043,841403,1,170438414031,Block Group 1,G5030,S,1286384,7410,+41.8646714,-088.1510559,"POLYGON ((-88.16152 41.8711, -88.1615 41.87114..."
2,17,039,971400,1,170399714001,Block Group 1,G5030,S,119380410,2350925,+40.2200603,-088.6970357,"POLYGON ((-88.74649 40.22437, -88.74647 40.226..."
3,17,039,971500,3,170399715003,Block Group 3,G5030,S,188071202,146822,+40.0975029,-088.8379859,"POLYGON ((-88.96666 40.12775, -88.96646 40.127..."
4,17,039,971500,2,170399715002,Block Group 2,G5030,S,197912862,16571541,+40.1992695,-088.8289742,"POLYGON ((-88.96348 40.21375, -88.96332 40.213..."


In [20]:
from shapely.geometry import Point

nei_gdf = gpd.GeoDataFrame(
    nei,
    geometry=gpd.points_from_xy(nei["site longitude"], nei["site latitude"]),
    crs="EPSG:4326"   # WGS84
)

nei_gdf.head()

,state,fips state code,tribal name,fips code,county,eis facility id,program system code,agency facility id,tri facility id,company name,...,pollutant type(s),hap type,total emissions,emissions uom,data set,outlier minimum,outlier maximum,outlier?,national maximum,geometry
26771,IL,17.0,NaN,17019,Champaign,558811,ILEPA,019813AAA,NaN,NaN,...,CAP,NaN,78.318483,TON,2021NEI_FinalV2,43.0,684.0,No,NaN,POINT (-88.4153 40.2844)
26785,IL,17.0,NaN,17031,Cook,558911,ILEPA,031012ADF,NaN,Na,...,CAP,NaN,1.006086,TON,2021NEI_FinalV2,NaN,NaN,No,1000.0,POINT (-87.80166 41.77294)
26857,IL,17.0,NaN,17031,Cook,559311,ILEPA,031006AAC,60501WNSCR5824S,Na,...,CAP,NaN,27.900600,TON,2021NEI_FinalV2,16.0,63.0,No,NaN,POINT (-87.81816 41.78473)
26893,IL,17.0,NaN,17031,Cook,559511,ILEPA,031012AEA,NaN,Na,...,CAP,NaN,6.143930,TON,2021NEI_FinalV2,NaN,NaN,No,1000.0,POINT (-87.83185 41.76796)
26908,IL,17.0,NaN,17011,Bureau,559711,ILEPA,011085ABC,61356SCHLG121WR,Na,...,CAP,NaN,1.370200,TON,2021NEI_FinalV2,NaN,NaN,No,1000.0,POINT (-89.46794 41.38699)


In [21]:
nei = gpd.sjoin(
    nei_gdf,
    bg[["GEOID", "geometry"]],
    how="left",
    predicate="within"
)

/var/tmp/ipykernel_30561/2741990392.py:1: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: EPSG:4326
Right CRS: EPSG:4269

  nei = gpd.sjoin(


In [22]:
nei["GEOID10"] = nei["GEOID"]

nei[["site latitude", "site longitude", "GEOID10"]].head()

,site latitude,site longitude,GEOID10
26771,40.284400,-88.415300,170190105003
26785,41.772942,-87.801655,170318205021
26857,41.784732,-87.818156,170318204001
26893,41.767962,-87.831849,170318205011
26908,41.386988,-89.467945,170119649002


In [23]:
nei["GEOID10"].isna().sum()

np.int64(0)

In [24]:
print("Total rows:", len(nei))
print("Matched GEOIDs:", nei["GEOID"].notna().sum())
print("Missing GEOIDs:", nei["GEOID"].isna().sum())

Total rows: 2319
Matched GEOIDs: 2319
Missing GEOIDs: 0


## Add Log Emissions 

In [25]:
nei["log_emissions"] = np.log(nei["total emissions"] + 1)

### Final Cleaning

In [26]:
print("Shape:", nei.shape)
print("\nColumns:\n", nei.columns)
print("\nMissing values:\n", nei.isnull().sum().sort_values(ascending=False).head(15))

Shape: (2319, 38)

Columns:
 Index(['state', 'fips state code', 'tribal name', 'fips code', 'county',
       'eis facility id', 'program system code', 'agency facility id',
       'tri facility id', 'company name', 'site name', 'primary naics code',
       'primary naics description', 'facility source type', 'site latitude',
       'site longitude', 'address', 'city', 'zip code', 'postal abbreviation',
       'reporting period', 'emissions operating type', 'pollutant code',
       'pollutant desc', 'pollutant type(s)', 'hap type', 'total emissions',
       'emissions uom', 'data set', 'outlier minimum', 'outlier maximum',
       'outlier?', 'national maximum', 'geometry', 'index_right', 'GEOID',
       'GEOID10', 'log_emissions'],
      dtype='object')

Missing values:
 tribal name             2319
hap type                2319
outlier minimum         2116
outlier maximum         2106
tri facility id         1933
facility source type    1062
company name             751
national maximum

In [27]:
nei_clean = nei[[
    "eis facility id",
    "company name",
    "site name",
    "primary naics code",
    "primary naics description",
    "facility source type",
    "site latitude",
    "site longitude",
    "GEOID10",
    "total emissions",
    "log_emissions"
]].copy()

print("Final shape:", nei_clean.shape)
nei_clean.head()

Final shape: (2319, 11)


,eis facility id,company name,site name,primary naics code,primary naics description,facility source type,site latitude,site longitude,GEOID10,total emissions,log_emissions
26771,558811,NaN,Peoples Gas Light & Coke Co,486210,Pipeline Transportation of Natural Gas,NaN,40.284400,-88.415300,170190105003,78.318483,4.373471
26785,558911,Na,Polartech a div of Italmatch SC LLC,325613,Surface Active Agent Manufacturing,NaN,41.772942,-87.801655,170318205021,1.006086,0.696186
26857,559311,Na,Owens-Corning Corp,324122,Asphalt Shingle and Coating Materials Manufact...,Hot Mix Asphalt Plant,41.784732,-87.818156,170318204001,27.900600,3.363862
26893,559511,Na,Kinder Morgan Liquid Terminals LP,493190,Other Warehousing and Storage,NaN,41.767962,-87.831849,170318205011,6.143930,1.966263
26908,559711,Na,LCN Closers Division of Schlage Lock Co LLC,332510,Hardware Manufacturing,NaN,41.386988,-89.467945,170119649002,1.370200,0.862974


## Save Datasets

In [28]:
# Save Data for Analysis
import gcsfs

fs = gcsfs.GCSFileSystem()

with fs.open(main_path_an, "w") as f:
    nei.to_csv(f, index=False)

print("Analysis dataset saved to GCS.")

Analysis dataset saved to GCS.


In [29]:
import gcsfs

fs = gcsfs.GCSFileSystem()

with fs.open(main_path_cl, "w") as f:
    nei_clean.to_csv(f, index=False)

print("Analysis dataset saved to GCS.")

Analysis dataset saved to GCS.
